# 04 ML Classical Algorithms — Baseline

**Baseline group.** Trains regressors using only classical network features (DebtRank, PageRank, centrality measures) with no embedding input. Used as the reference point for all embedding experiments.

In [1]:
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'src').exists() and (path / 'requirements.txt').exists():
            return path
    raise FileNotFoundError('Project root not found.')


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
from src.models.ml_train_and_store import (
    ModelTrainer,
    load_classical_dataset,
    make_pipeline,
)

pd.set_option("display.max_columns", 200)
PROJECT_ROOT = find_project_root()

## Load Dataset

In [2]:
print(PROJECT_ROOT)

C:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis


In [3]:
df, feature_cols = load_classical_dataset(PROJECT_ROOT)
df.shape, feature_cols

((145536, 90),
 ['degree_centrality_total',
  'weighted_degree_total',
  'betweenness_centrality',
  'closeness_centrality',
  'eigenvector_centrality',
  'pagerank',
  'debtrank'])

In [4]:
trainer = ModelTrainer(
    df=df,
    feature_cols=feature_cols,
    target_col="log_systemic_risk_label",
)

trainer.train_df.shape, trainer.val_df.shape, trainer.test_df.shape

((109152, 90), (18192, 90), (13644, 90))

## Define Models

In [5]:
DISPLAY_COLS = ["model", "train_mae", "validation_mae", "train_rmse", "validation_rmse"]
TOP1_COLS    = ["model", "train_top1_mae", "validation_top1_mae", "train_top1_rmse", "validation_top1_rmse"]

candidate_models = {
    "Linear Regression": make_pipeline(LinearRegression()),
    "Ridge":             make_pipeline(Ridge(alpha=1.0)),
    "MLP":               make_pipeline(MLPRegressor(hidden_layer_sizes=(32,63,16), max_iter=300, activation='relu', learning_rate="adaptive",learning_rate_init=0.001 , early_stopping=True, n_iter_no_change=3, random_state=42)),
    "Random Forest":     make_pipeline(RandomForestRegressor(n_estimators=100, random_state=42)),
    "Gradient Boosting": make_pipeline(HistGradientBoostingRegressor(max_iter=200, random_state=42)),
    "XGBoost":           make_pipeline(XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42)),
}

list(candidate_models)

['Linear Regression',
 'Ridge',
 'MLP',
 'Random Forest',
 'Gradient Boosting',
 'XGBoost']

## Train And Store

In [6]:
trainer.train_all(candidate_models)
trainer.leaderboard()[DISPLAY_COLS]

c:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis\.venv\Lib\site-packages\sklearn\linear_model\_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 1.9798088627289653e-77.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


,model,train_mae,validation_mae,train_rmse,validation_rmse
0,Random Forest,0.003502,0.014545,0.021117,0.08289
1,Gradient Boosting,0.009013,0.014706,0.053364,0.083878
2,XGBoost,0.00797,0.015085,0.044758,0.084803
3,Ridge,0.024997,0.037311,0.095156,0.150539
4,Linear Regression,0.040509,0.051685,0.176157,0.23651
5,MLP,45479499213914985950347264.0,9858213852915.951172,8206783496658376125027713024.0,993024298422562.125


## Hyperparameter Tuning

Uses `RandomizedSearchCV` with `PredefinedSplit` so the temporal train/val boundary is respected — train rows are always used for fitting, val rows always for scoring.

In [7]:
def tune(trainer, base_model, param_distributions, name, n_iter=40):
    X = pd.concat([trainer.train_df[trainer.feature_cols], trainer.val_df[trainer.feature_cols]])
    y = pd.concat([trainer.train_df[trainer.target_col],   trainer.val_df[trainer.target_col]])
    split_idx = np.concatenate([
        np.full(len(trainer.train_df), -1),
        np.zeros(len(trainer.val_df), dtype=int),
    ])
    search = RandomizedSearchCV(
        base_model,
        param_distributions,
        n_iter=n_iter,
        cv=PredefinedSplit(split_idx),
        scoring="neg_root_mean_squared_error",
        random_state=42,
        n_jobs=-1,
    )
    search.fit(X, y)
    trainer.train(search.best_estimator_, name=name)
    trainer.best_params[name] = search.best_params_
    return search.best_params_

RF_PARAMS = {
    "model__n_estimators":      [100, 200, 300, 400, 500, 600],
    "model__max_depth":         [None, 5, 10, 15, 20, 30],
    "model__min_samples_leaf":  [1, 2, 5, 10, 15, 20],
    "model__min_samples_split": [2, 5, 10, 15, 20],
    "model__max_features":      ["sqrt", "log2", 0.5, 0.8, 1.0],
}
GB_PARAMS = {
    "model__max_iter":          [100, 200, 300, 400, 500, 600],
    "model__max_depth":         [3, 4, 5, 6, 8, None],
    "model__learning_rate":     [0.005, 0.01, 0.05, 0.1, 0.2, 0.3],
    "model__min_samples_leaf":  [5, 10, 20, 50, 100],
    "model__l2_regularization": [1e-4, 1e-3, 1e-2, 0.1, 1.0],
    "model__max_leaf_nodes":    [15, 20, 30, 40, 50, 60],
    "model__max_bins":          [64, 128, 255],
}
XGB_PARAMS = {
    "model__n_estimators":     [100, 200, 400, 600, 800],
    "model__max_depth":        [3, 4, 5, 6, 8, 10],
    "model__learning_rate":    [0.005, 0.01, 0.05, 0.1, 0.2, 0.3],
    "model__subsample":        [0.6, 0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.5, 0.6, 0.7, 0.8, 1.0],
    "model__min_child_weight": [1, 2, 5, 10],
    "model__gamma":            [0, 0.1, 0.5, 1.0, 2.0],
    "model__reg_alpha":        [1e-5, 1e-4, 1e-3, 1e-2, 0.1, 1.0],
    "model__reg_lambda":       [1e-5, 1e-4, 1e-3, 1e-2, 0.1, 1.0, 2.0],
}
MLP_PARAMS = {
    "model__hidden_layer_sizes": [(64,), (128,), (256,), (128, 64), (256, 128), (128, 64, 32), (256, 128, 64)],
    "model__activation":         ["relu", "tanh"],
    "model__alpha":              [1e-5, 1e-4, 1e-3, 1e-2, 0.1],
    "model__learning_rate_init": [1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 0.05],
    "model__learning_rate":      ["constant", "adaptive"],
    "model__batch_size":         [32, 64, 128, "auto"],
}

In [8]:
tune(trainer, make_pipeline(RandomForestRegressor(random_state=42)),     RF_PARAMS,  "Random Forest (tuned)")
tune(trainer, make_pipeline(HistGradientBoostingRegressor(random_state=42)), GB_PARAMS, "Gradient Boosting (tuned)")
tune(trainer, make_pipeline(XGBRegressor(random_state=42)),              XGB_PARAMS, "XGBoost (tuned)")
tune(trainer, make_pipeline(MLPRegressor(max_iter=500, early_stopping=True, n_iter_no_change=20, random_state=42)), MLP_PARAMS, "MLP (tuned)")
trainer.leaderboard()[DISPLAY_COLS]
trainer.leaderboard()[TOP1_COLS]

,model,train_mae,validation_mae,train_rmse,validation_rmse
0,Random Forest (tuned),0.005265,0.014155,0.030863,0.079255
1,Random Forest,0.003502,0.014545,0.021117,0.08289
2,Gradient Boosting (tuned),0.009752,0.014558,0.059807,0.083328
3,Gradient Boosting,0.009013,0.014706,0.053364,0.083878
4,XGBoost,0.00797,0.015085,0.044758,0.084803
5,XGBoost (tuned),0.00827,0.015595,0.04583,0.086197
6,Ridge,0.024997,0.037311,0.095156,0.150539
7,MLP (tuned),0.042994,0.051003,0.150367,0.196757
8,Linear Regression,0.040509,0.051685,0.176157,0.23651
9,MLP,45479499213914985950347264.0,9858213852915.951172,8206783496658376125027713024.0,993024298422562.125


In [9]:
trainer.leaderboard()[TOP1_COLS]

,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Random Forest (tuned),0.147939,0.450312,0.185832,0.516713
1,Random Forest,0.107731,0.466073,0.132253,0.543008
2,Gradient Boosting (tuned),0.319655,0.45164,0.401638,0.540998
3,Gradient Boosting,0.286214,0.468694,0.354233,0.549957
4,XGBoost,0.222599,0.48199,0.27845,0.557066
5,XGBoost (tuned),0.231217,0.498047,0.284611,0.566999
6,Ridge,0.698042,0.987142,0.778696,1.050406
7,MLP (tuned),0.90917,1.168231,1.102824,1.343424
8,Linear Regression,1.360545,1.483301,1.547842,1.698316
9,MLP,2723660772939471120781803520.0,523668670358275.375,64569361650735833232741236736.0,7252969948393154.0


## Optional — Inspect Best Hyperparameters / Save Models

In [10]:
# Display best hyperparameters found during tuning — run this cell when you want to inspect them
if trainer.best_params:
    for model_name, params in trainer.best_params.items():
        print(f"\n{model_name}:")
        for k, v in params.items():
            print(f"  {k.replace('model__', '')}: {v}")


Random Forest (tuned):
  n_estimators: 300
  min_samples_split: 2
  min_samples_leaf: 2
  max_features: 0.5
  max_depth: 15

Gradient Boosting (tuned):
  min_samples_leaf: 20
  max_leaf_nodes: 20
  max_iter: 300
  max_depth: 8
  max_bins: 128
  learning_rate: 0.2
  l2_regularization: 0.01

XGBoost (tuned):
  subsample: 0.6
  reg_lambda: 1.0
  reg_alpha: 0.01
  n_estimators: 100
  min_child_weight: 1
  max_depth: 6
  learning_rate: 0.1
  gamma: 0
  colsample_bytree: 0.8

MLP (tuned):
  learning_rate_init: 0.01
  learning_rate: adaptive
  hidden_layer_sizes: (128, 64)
  batch_size: auto
  alpha: 0.001
  activation: tanh


In [11]:
# Save specific models to disk — run this cell when you want to persist them
from src.models.ml_train_and_store import load_model

SAVE_DIR = PROJECT_ROOT / "src" / "models" / "train_saved" / "04_a"

best_name = 'Gradient Boosting (tuned)'
path = trainer.save_model(best_name, SAVE_DIR)

# To reload later:
# model = load_model(path)

Saved 'Gradient Boosting (tuned)' → C:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis\src\models\train_saved\04_a\Gradient_Boosting_(tuned).joblib
